# 26 — RAG Fusion

Generate N query paraphrases in **parallel**, retrieve independently for each, then merge with **Reciprocal Rank Fusion** for higher recall than any single-query RAG.

**What you'll learn**
- Why a single query is a single point of failure in retrieval
- How to fan out N parallel retrieval branches using the LangGraph Send API
- `Annotated[list, operator.add]` — how parallel workers merge results into shared state
- RRF scoring: `score = Σ 1/(k + rank_i)` across all ranked lists
- The `generate_variants → [fan-out] → retrieve_variant → fuse → generate` pipeline

**Companion examples:** 28-parallel-subgraph (Send API deep dive), 25-adaptive-rag (routing), 22-hybrid-search-rag (ensemble retrieval)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langchain-chroma chromadb python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Knowledge base + RRF constants ────────────────────────────────────────
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

DOCS = [
    Document(page_content="LangGraph is a library for building stateful, multi-actor LLM applications using graph-based workflows."),
    Document(page_content="LangChain provides tools for building LLM applications: chains, agents, and retrieval systems."),
    Document(page_content="Vector databases store embeddings for semantic search. Popular options: Chroma, Pinecone, Weaviate."),
    Document(page_content="RAG (Retrieval-Augmented Generation) grounds LLM responses in external documents to reduce hallucination."),
    Document(page_content="Reciprocal Rank Fusion (RRF) merges ranked lists: score = sum(1 / (k + rank)). Lower rank = higher score."),
    Document(page_content="The LangGraph Send API allows parallel node execution by dispatching the same event to multiple workers."),
    Document(page_content="Query expansion generates multiple paraphrases of a question to improve recall across embedding representations."),
    Document(page_content="BM25 ranks documents by term frequency and inverse document frequency — keyword-based retrieval."),
    Document(page_content="Hybrid search combines vector similarity and BM25, merging results with ensemble or RRF methods."),
    Document(page_content="Self-RAG grades retrieved documents and uses reflection tokens to decide when retrieval is needed."),
]

NUM_VARIANTS = 4   # query paraphrases to fan out
K = 3              # top-k docs per variant retrieval
RRF_K = 60         # RRF constant (standard value from Lawrence 2024)

vectorstore = Chroma.from_documents(
    DOCS, OpenAIEmbeddings(model="text-embedding-3-small"), collection_name="rag_fusion"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": K})

print(f"KB ready: {len(DOCS)} docs | NUM_VARIANTS={NUM_VARIANTS} | K={K} | RRF_K={RRF_K}")

In [ ]:
# ── 4. RRF merge function ─────────────────────────────────────────────────────
# A doc appearing high in MULTIPLE ranked lists gets a high fused score.
# A doc appearing in only one list ranks lower, even if it's rank 1 there.

def rrf_merge(all_ranked_lists: list[list[str]], k: int = RRF_K) -> list[str]:
    scores: dict[str, float] = {}
    for ranked_list in all_ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.__getitem__, reverse=True)


# Quick demo with toy ranked lists
demo = [
    ["doc_A", "doc_B", "doc_C"],  # variant-1 ranking
    ["doc_B", "doc_A", "doc_D"],  # variant-2 ranking
]
print("Fused:", rrf_merge(demo))  # doc_A and doc_B appear in both → higher scores

In [ ]:
# ── 5. Build the RAG Fusion graph ─────────────────────────────────────────────
# Two key LangGraph primitives:
#   Send("retrieve_variant", {...}) — each call dispatches one parallel worker
#   Annotated[list, operator.add]  — workers' results are appended together automatically
from operator import add
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.constants import Send
from langgraph.graph import END, START, StateGraph


class VariantState(TypedDict):
    variant: str
    ranked_docs: list[str]


class RAGFusionState(TypedDict):
    question: str
    variants: list[str]
    ranked_docs: Annotated[list[list[str]], add]  # each worker appends one ranked list
    fused_docs: list[str]
    answer: str


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def generate_variants(state: RAGFusionState) -> dict:
    response = llm.invoke([
        SystemMessage(
            f"Generate exactly {NUM_VARIANTS} alternative phrasings of the question. "
            "Each should approach the topic from a different angle. "
            f"Return only {NUM_VARIANTS} questions, one per line, no numbering."
        ),
        HumanMessage(state["question"]),
    ])
    raw = response.content.strip().split("\n")
    variants = [q.strip() for q in raw if q.strip()][:NUM_VARIANTS]
    if state["question"] not in variants:
        variants = [state["question"]] + variants[: NUM_VARIANTS - 1]
    return {"variants": variants}


def fan_out(state: RAGFusionState):
    """Return one Send per variant — all retrieve_variant calls run in parallel."""
    return [Send("retrieve_variant", {"variant": v, "ranked_docs": []}) for v in state["variants"]]


def retrieve_variant(state: VariantState) -> dict:
    docs = retriever.invoke(state["variant"])
    return {"ranked_docs": [d.page_content for d in docs]}


def fuse(state: RAGFusionState) -> dict:
    return {"fused_docs": rrf_merge(state["ranked_docs"])}


def generate(state: RAGFusionState) -> dict:
    context = "\n\n".join(state["fused_docs"])
    response = llm.invoke([
        SystemMessage(f"Answer using only the fused context below.\n\nContext:\n{context}"),
        HumanMessage(state["question"]),
    ])
    return {"answer": response.content}


graph = StateGraph(RAGFusionState)
graph.add_node("generate_variants", generate_variants)
graph.add_node("retrieve_variant", retrieve_variant)
graph.add_node("fuse", fuse)
graph.add_node("generate", generate)
graph.add_edge(START, "generate_variants")
graph.add_conditional_edges("generate_variants", fan_out, ["retrieve_variant"])
graph.add_edge("retrieve_variant", "fuse")
graph.add_edge("fuse", "generate")
graph.add_edge("generate", END)
app = graph.compile()

print("Graph: START → generate_variants → [N×retrieve_variant] → fuse → generate → END")

In [ ]:
# ── 6. Run RAG Fusion ─────────────────────────────────────────────────────────
QUESTIONS = [
    "How does RAG Fusion work?",
    "What is the LangGraph Send API used for?",
    "Explain Reciprocal Rank Fusion for combining search results",
]

for question in QUESTIONS:
    print(f"\nQ: {question}")
    result = app.invoke({"question": question, "variants": [], "ranked_docs": [], "fused_docs": [], "answer": ""})
    print(f"  Variants   : {len(result['variants'])}")
    print(f"  Fused docs : {len(result['fused_docs'])}")
    print(f"  Answer     : {result['answer'][:200]}")

## Exercises

**Exercise 1 — Inspect the variants:** Print `result['variants']` for one question. Do the paraphrases genuinely approach the topic from different angles, or are they too similar?

**Exercise 2 — Compare against single-query RAG:** Run `retriever.invoke(question)` directly and compare its top-3 docs against RAG Fusion's fused top-3. For which questions does fusion help most?

**Exercise 3 — Change NUM_VARIANTS:** Try `NUM_VARIANTS = 2` and `NUM_VARIANTS = 8`. How does the number of fused unique docs change? Where does adding more variants stop helping?

## What's next

- **28-parallel-subgraph** — the Send API map-reduce pattern in isolation (document summarisation)
- **25-adaptive-rag** — classify queries before retrieval to pick the cheapest strategy
- **27-self-rag** — decide WHETHER to retrieve at all using LLM reflection gates